# Day 45 — Too Many Columns

## Principal Component Analysis

**Boston University · Data Science Program**

---

Every model so far has taken whatever columns you handed it. Today is the first time we
ask whether you have **too many**, and what to do about it.

The whole of PCA is one idea, and you can see it in a picture:

> **Find the direction your data spreads out along the most. Use it as your new column.**

Everything else — eigenvectors, explained variance, scree plots — is machinery for doing
that reliably. We are going to build it from a picture first, then name the parts.

Run every cell. Change the numbers. Break something on purpose.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_wine, load_breast_cancer, load_digits

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED, GREY = "#7B8AA1", "#C9D2E0"
np.set_printoptions(precision=4, suppress=True)
print("ready")

---
# 1 · The problem

To look at data you plot one column against another. Count how many plots that is.

In [ ]:
for c in [2, 6, 13, 30, 64]:
    print(f"{c:3d} columns  ->  {c * (c - 1) // 2:5d} scatterplots")

Six columns is already fifteen plots. Sixty-four pixels is two thousand. Nobody opens
two thousand plots, so for anything wide you are flying blind.

But there is a second, better reason to want fewer columns, and it shows up as soon as
you look at the correlations.

### Our data: 300 students, six subject marks

Nobody has labelled anything. This is unsupervised learning — same as the last two days.

In [ ]:
SUBJECTS = ["Maths", "Physics", "Chemistry", "English", "History", "Art"]

def make_marks(n=300, seed=45):
    r = np.random.default_rng(seed)
    general = r.normal(0, 1, n)        # how good a student is overall
    leaning = r.normal(0, 1, n)        # science-minded (+) or arts-minded (-)
    W_GEN = np.array([9.0, 9.0, 8.5, 8.0, 7.5, 6.5])
    W_LEAN = np.array([7.5, 6.5, 5.0, -5.5, -6.5, -7.0])
    BASE = np.array([62.0, 60.0, 64.0, 68.0, 66.0, 70.0])
    NOISE = np.array([4.0, 4.5, 4.5, 4.0, 5.0, 6.0])
    M = (BASE + general[:, None] * W_GEN + leaning[:, None] * W_LEAN
         + r.normal(0, 1, (n, 6)) * NOISE)
    return np.clip(M, 0, 100), general, leaning

M, GENERAL, LEANING = make_marks()
marks = pd.DataFrame(M, columns=SUBJECTS)
print(marks.shape)
marks.head().round(1)

`general` and `leaning` are the two hidden numbers the marks were built from. **PCA will
never see them.** We keep them only so that at the end we can check whether PCA found
them on its own.

Now the correlations.

In [ ]:
corr = marks.corr()
print(corr.round(2))

Look at the block structure. **Maths / Physics / Chemistry** all correlate at 0.85 or
higher with each other. **English / History / Art** do the same. Across the two blocks,
the biggest correlation is 0.36.

Two columns correlating at 0.87 are largely **repeating each other**. You do not have six
independent pieces of information here — you have something closer to two.

That is what PCA is for: find those two, and write them down as columns.

---
# 2 · One direction at a time

Six columns is hard to picture, so start with two: **Maths and Physics**.

Standardise first — always, before PCA. We will see exactly why in section 5.

In [ ]:
Z = StandardScaler().fit_transform(M)
T = Z[:, [0, 1]]                       # Maths, Physics only

plt.figure(figsize=(4.6, 4.4))
plt.scatter(T[:, 0], T[:, 1], s=18, c=GREY, edgecolors="white", linewidths=.4)
plt.axhline(0, color="#E2E8F2"); plt.axvline(0, color="#E2E8F2")
plt.xlabel("Maths (standardised)"); plt.ylabel("Physics (standardised)")
plt.gca().set_aspect("equal"); plt.title("the cloud leans"); plt.show()

print("correlation:", round(np.corrcoef(T[:, 0], T[:, 1])[0, 1], 4))

### Projecting a student onto a direction

A **direction** is just two weights, `w = (w1, w2)`, with `w1² + w2² = 1`.

To project a student onto that direction, multiply and add:

$$p = w_1 x_1 + w_2 x_2$$

That is it. Two columns in, one number out. In numpy it is a single `@`.

In [ ]:
angle = 45
w = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
print("w =", w.round(4), "   length =", round(np.linalg.norm(w), 6))

student = np.array([1.2, 0.8])         # Maths +1.2, Physics +0.8
print(f"\nby hand : {w[0]:.3f} * {student[0]} + {w[1]:.3f} * {student[1]}"
      f" = {w[0] * student[0] + w[1] * student[1]:.4f}")
print(f"with @  : {student @ w:.4f}")

p = T @ w                              # every student at once
print("\nall 300 students projected:", p.shape, " first five:", p[:5].round(3))

### Which direction is best?

Here is the whole of PCA in one question:

> **Which direction spreads the students out the most?**

A direction that squashes everyone together has thrown your data away. A direction that
spreads them out has kept the differences between them — and the differences are the
information.

So: try every direction and measure the spread.

In [ ]:
angles = np.arange(0, 180, 0.25)
spreads = np.array([
    np.var(T @ np.array([np.cos(np.radians(a)), np.sin(np.radians(a))]), ddof=1)
    for a in angles
])

best = angles[spreads.argmax()]
print(f"best direction : {best:.1f}°   spread {spreads.max():.4f}")
print(f"worst direction: {angles[spreads.argmin()]:.1f}°   spread {spreads.min():.4f}")
print()
for a in [0, 45, 90, 135]:
    wa = np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])
    print(f"  {a:3d}° -> spread {np.var(T @ wa, ddof=1):.4f}")

plt.figure(figsize=(6.2, 3.2))
plt.plot(angles, spreads, color=BLUE, lw=2.5)
plt.axhline(1.0, color=MUTED, ls=":", lw=1)
plt.axvline(best, color=TEAL, ls="--", lw=1.4)
plt.text(best + 4, spreads.max(), f"  {best:.0f}° -> {spreads.max():.2f}",
         color=TEAL, fontweight="bold", va="center")
plt.xticks([0, 45, 90, 135, 180]); plt.xlabel("direction"); plt.ylabel("spread")
plt.title("every direction, measured"); plt.tight_layout(); plt.show()

Read those four numbers carefully:

- **0°** is Maths alone. Spread **1.00** — of course, we standardised it.
- **90°** is Physics alone. Spread **1.00** too.
- **45°** mixes them equally. Spread **1.88** — **87% more than either column on its own.**
- **135°** is the other diagonal. Spread **0.13** — that direction has almost nothing to say.

**A mixture of two columns beat both of the columns.** That is not obvious, and it is the
entire reason PCA exists.

---
# 3 · The formula

Sweeping 720 angles worked, but it is clumsy and it does not generalise — in 64
dimensions there is no "angle" to sweep.

There is a formula. The spread along direction `w` is

$$\text{spread} = w_1^2 C_{11} + w_2^2 C_{22} + 2 w_1 w_2 C_{12}$$

where **C is the covariance matrix** — a small table computed from the data *once*:

- `C11` is how much column 1 varies on its own
- `C22` is how much column 2 varies on its own
- `C12` is how much the two move together

Change `w` and the spread changes. **C never does.**

In [ ]:
C = np.cov(T, rowvar=False)
print("covariance matrix C:")
print(C.round(4))
print(f"\nC11 = {C[0,0]:.4f}   C22 = {C[1,1]:.4f}   C12 = {C[0,1]:.4f}")

# the formula, written out by hand at 45 degrees
w45 = np.array([np.cos(np.radians(45)), np.sin(np.radians(45))])
by_hand = w45[0]**2 * C[0,0] + w45[1]**2 * C[1,1] + 2 * w45[0] * w45[1] * C[0,1]

print(f"\nby hand        : {by_hand:.4f}")
print(f"as w.T @ C @ w : {w45 @ C @ w45:.4f}")
print(f"from the sweep : {spreads[angles == 45][0]:.4f}")

All three agree to four decimals. In matrix form the formula is just

$$\text{spread} = w^{T} C\, w$$

which is the version that works for any number of columns.

### The direction that wins has a name

Maximising $w^{T}Cw$ over all unit vectors `w` is a solved problem in linear algebra. The
answer is the **top eigenvector** of C, and the spread you get is the matching
**eigenvalue**, λ.

Those two words sound intimidating. They are not:

| Word | What it actually is here |
|---|---|
| **eigenvector** | the direction that won the sweep |
| **eigenvalue** | the height of the peak on that curve |

Let numpy find them.

In [ ]:
evals, evecs = np.linalg.eigh(C)
order = evals.argsort()[::-1]                 # biggest first
evals, evecs = evals[order], evecs[:, order]

pc1, pc2 = evecs[:, 0], evecs[:, 1]
ang = np.degrees(np.arctan2(abs(pc1[1]), abs(pc1[0])))

print(f"eigenvalues : {evals.round(4)}")
print(f"PC1 direction: {pc1.round(4)}   = {ang:.1f}°")
print(f"PC2 direction: {pc2.round(4)}")
print()
print(f"sweep said   : {best:.1f}°  with spread {spreads.max():.4f}")
print(f"eigen says   : {ang:.1f}°  with spread {evals[0]:.4f}")
print()
print("PC1 . PC2 =", round(float(pc1 @ pc2), 12), " (they are at right angles)")
print("length of PC1 =", round(float(np.linalg.norm(pc1)), 6))

The sweep and the eigenvector agree exactly. **You never need to sweep** — but knowing
that is what the eigenvector *is* is the difference between using PCA and trusting it.

Two facts worth noticing:

- **PC2 is perpendicular to PC1** (their dot product is 0). That is forced, and it is why
  components never repeat each other: PC2 is only allowed to describe what PC1 could not.
- **The eigenvalues add up to the total spread.** Watch:

In [ ]:
print(f"eigenvalues sum to : {evals.sum():.4f}")
print(f"total variance     : {np.trace(C):.4f}   (C11 + C22)")
print()
print("PCA does not create or destroy spread. It only rearranges where it sits.")

### Explained variance ratio

Since the λ add up to the total, each one's **share** is a clean fraction:

$$\text{share of PC}_i = \frac{\lambda_i}{\lambda_1 + \lambda_2 + \cdots}$$

scikit-learn calls this `explained_variance_ratio_`, and it is the number you will quote
most often.

In [ ]:
p2 = PCA().fit(T)
print("sklearn eigenvalues  :", p2.explained_variance_.round(4))
print("our eigenvalues      :", evals.round(4))
print()
print("explained_variance_ratio_:", p2.explained_variance_ratio_.round(4))
print("by hand                  :", (evals / evals.sum()).round(4))
print()
print(f"PC1 holds {p2.explained_variance_ratio_[0] * 100:.1f}% of everything "
      f"these two columns contained.")

**93.5% in one column.** Replace Maths and Physics with a single number and you keep
93.5% of what made students different. You lose 6.5%, permanently — PCA is **lossy**, and
the loss is the price of the compression.

Let us see what that loss looks like.

In [ ]:
proj = T @ pc1
foot = proj[:, None] * pc1[None, :]           # where each student lands on the line

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (pts, ttl) in zip(axes, [(T, "before"), (foot, "after: everyone on one line")]):
    ax.plot([-3.5*pc1[0], 3.5*pc1[0]], [-3.5*pc1[1], 3.5*pc1[1]], color=BLUE, lw=2.4)
    ax.scatter(pts[:, 0], pts[:, 1], s=20, c=AMBER if ttl.startswith("after") else GREY,
               edgecolors="white", linewidths=.4)
    ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.4, 3.4); ax.set_aspect("equal")
    ax.set_title(ttl); ax.set_xlabel("Maths"); ax.set_ylabel("Physics")
plt.tight_layout(); plt.show()

lost = T - foot
print(f"spread kept : {np.var(proj, ddof=1):.4f}")
print(f"spread lost : {np.var(T @ pc2, ddof=1):.4f}")
print(f"the student who moved furthest moved {np.linalg.norm(lost, axis=1).max():.2f}")

---
# 4 · All six columns

Same idea, more of it. Now `w` has six weights instead of two, and there are six
components instead of two.

In [ ]:
pca6 = PCA().fit(Z)
evr = pca6.explained_variance_ratio_

for i, (v, c) in enumerate(zip(evr, np.cumsum(evr)), 1):
    bar = "#" * int(round(v * 60))
    print(f"PC{i}  {v*100:5.1f}%  (running total {c*100:5.1f}%)  {bar}")

**PC1 and PC2 together hold 88.9%.** Six columns have become two, and almost nothing was
lost. Compare that with the two-column case, where dropping to one cost 6.5% — the more
your columns repeat each other, the better this trade gets.

### What do PC1 and PC2 mean?

A component is a set of weights, one per original column. Those weights are called
**loadings**, and they are how a component gets a name.

In [ ]:
load = pd.DataFrame(pca6.components_[:2].T, index=SUBJECTS, columns=["PC1", "PC2"])
print(load.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.2))
for ax, c in zip(axes, ["PC1", "PC2"]):
    vals = load[c].values
    ax.barh(range(6), vals, color=[TEAL if v > 0 else CORAL for v in vals], height=.62)
    ax.set_yticks(range(6)); ax.set_yticklabels(SUBJECTS); ax.invert_yaxis()
    ax.axvline(0, color="#131F33", lw=1.2); ax.set_xlim(-.62, .62); ax.set_title(c)
plt.tight_layout(); plt.show()

Read the signs:

- **PC1 — every loading is positive**, all between 0.33 and 0.47. A student with a high
  PC1 did well in *everything*. So PC1 means **overall ability**.
- **PC2 — the signs split.** Maths, Physics, Chemistry negative; English, History, Art
  positive. A high PC2 means **arts over science**.

Nobody told PCA about "ability" or "leaning". It read six columns of marks and those two
ideas fell out.

### Now the check we have been saving

Remember `general` and `leaning` — the two hidden numbers that actually built this data,
which PCA has never seen. How close did it get?

In [ ]:
S = pca6.transform(Z)

print(f"PC1 vs hidden 'general' : {abs(np.corrcoef(S[:,0], GENERAL)[0,1]):.4f}")
print(f"PC1 vs hidden 'leaning' : {abs(np.corrcoef(S[:,0], LEANING)[0,1]):.4f}")
print(f"PC2 vs hidden 'general' : {abs(np.corrcoef(S[:,1], GENERAL)[0,1]):.4f}")
print(f"PC2 vs hidden 'leaning' : {abs(np.corrcoef(S[:,1], LEANING)[0,1]):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
for ax, (vals, ttl) in zip(axes, [(GENERAL, "coloured by hidden 'general'"),
                                  (LEANING, "coloured by hidden 'leaning'")]):
    sc = ax.scatter(S[:, 0], S[:, 1], c=vals, cmap="coolwarm", s=20,
                    edgecolors="white", linewidths=.3)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(ttl)
plt.tight_layout(); plt.show()

**0.971 and 0.965.**

PCA recovered both hidden factors almost exactly, from nothing but the marks. The left
plot shows `general` running left-to-right along PC1; the right shows `leaning` running
bottom-to-top along PC2.

This is the best case — we built the data this way on purpose. Real data is messier. But
it shows what PCA is *for*: when several columns are driven by a few underlying things,
PCA finds those things.

---
# 5 · Scaling, and the mistake that silently ruins PCA

PCA chases **spread**. A column measured in hundreds has more spread than a column
measured in tenths — no matter what the columns mean.

So if you forget to scale, PCA ranks your columns **by their units**.

In [ ]:
WX, WY = load_wine(return_X_y=True)
WN = load_wine().feature_names
sds = WX.std(axis=0, ddof=1)

print("standard deviation of each wine column:")
for n, s in sorted(zip(WN, sds), key=lambda t: -t[1]):
    print(f"  {n:32} {s:10.4f}")
print(f"\nbiggest / smallest = {sds.max() / sds.min():.0f}x")

In [ ]:
pu = PCA().fit(WX)                               # NO scaling
print("WITHOUT scaling")
print(f"  PC1 holds {pu.explained_variance_ratio_[0]*100:.1f}% of the variance")
top = np.abs(pu.components_[0]).argsort()[::-1]
for i in top[:3]:
    print(f"    loading on {WN[i]:28} {pu.components_[0][i]:+.4f}")

WZ = StandardScaler().fit_transform(WX)
ps = PCA().fit(WZ)                               # WITH scaling
print("\nAFTER scaling")
print(f"  PC1 holds {ps.explained_variance_ratio_[0]*100:.1f}% of the variance")
top = np.abs(ps.components_[0]).argsort()[::-1]
for i in top[:3]:
    print(f"    loading on {WN[i]:28} {ps.components_[0][i]:+.4f}")

Unscaled, **PC1 is 99.8% of the variance and its loading on `proline` is 0.9998** — the
next column gets 0.018. PC1 is not a component at all. It is the proline column wearing a
hat, because proline is measured in the hundreds and everything else is not.

A 99.8% explained variance ratio *looks* like a triumph. Here it is a bug.

Does it matter downstream? Measure it.

In [ ]:
for tag, D in [("unscaled", WX), ("scaled  ", WZ)]:
    two = PCA(2).fit_transform(D)
    a = cross_val_score(LogisticRegression(max_iter=5000), two, WY, cv=5).mean()
    print(f"{tag}: accuracy from 2 components = {a:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
for ax, (tag, D) in zip(axes, [("without scaling", WX), ("after scaling", WZ)]):
    S2 = PCA(2).fit_transform(D)
    for c, col in zip([0, 1, 2], [TEAL, CORAL, VIOLET]):
        ax.scatter(S2[WY == c, 0], S2[WY == c, 1], s=22, c=col,
                   edgecolors="white", linewidths=.4, label=f"cultivar {c+1}")
    ax.set_title(tag); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**0.698 against 0.955.** Forgetting `StandardScaler` cost a quarter of the accuracy, and
nothing in the output warned you — the unscaled run reported a *higher* explained
variance.

> **Rule: `StandardScaler()` then `PCA()`. Every time.**

---
# 6 · How many components do you keep?

The standard tool is a **scree plot**: each component's share as a bar, with the running
total on top.

In [ ]:
evrw = ps.explained_variance_ratio_
cum = np.cumsum(evrw)

plt.figure(figsize=(6.4, 3.4))
plt.bar(range(1, 14), evrw * 100, color=BLUE, width=.62)
plt.plot(range(1, 14), cum * 100, color=CORAL, lw=2.4, marker="o", ms=5)
plt.axhline(90, color=TEAL, ls="--", lw=1.4)
plt.text(0.7, 92, "90%", color=TEAL, fontweight="bold")
plt.xticks(range(1, 14)); plt.xlabel("component"); plt.ylabel("% of the spread")
plt.title("wine: 13 columns"); plt.tight_layout(); plt.show()

for thr in [0.80, 0.90, 0.95]:
    print(f"to reach {thr:.0%}: keep {np.argmax(cum >= thr) + 1} components")

Wine needs **8 of its 13** components to reach 90%. That is a dataset whose columns do
*not* repeat each other much — compare the marks data, where 2 of 6 gave you 88.9%.

### But a variance threshold is a guide, not an answer

The usual advice is "keep 95% of the variance". Let us test that advice against something
that actually matters — a model's accuracy.

In [ ]:
DX, DY = load_digits(return_X_y=True)
DZ = StandardScaler().fit_transform(DX)
Xtr, Xte, ytr, yte = train_test_split(DZ, DY, test_size=0.3, random_state=0, stratify=DY)

full = accuracy_score(yte, KNeighborsClassifier(5).fit(Xtr, ytr).predict(Xte))
cumd = np.cumsum(PCA().fit(DZ).explained_variance_ratio_)

print(f"{'components':>11}{'variance kept':>16}{'accuracy':>11}")
for k in [2, 5, 10, 20, 30, 40]:
    pk = PCA(k, random_state=0).fit(Xtr)
    a = accuracy_score(yte, KNeighborsClassifier(5)
                       .fit(pk.transform(Xtr), ytr).predict(pk.transform(Xte)))
    print(f"{k:>11}{cumd[k-1]:>15.1%}{a:>11.4f}")
print(f"{'all 64':>11}{1.0:>15.1%}{full:>11.4f}")
print(f"\n95% of the variance needs {np.argmax(cumd >= 0.95) + 1} components")

Read the two ends against each other:

- **20 components score 0.970 — exactly what all 64 columns score.** 44 columns gone, and
  the model cannot tell. They hold only 79.3% of the variance.
- **The 95% rule asks for 40 components.** Twice as many, for no extra accuracy at all.

Variance and accuracy are not the same thing, and only one of them is your actual goal.

> **Use a threshold to pick a range to try. Then test, and let the task decide.**

---
# 7 · What you threw away

"Keeps 89% of the variance" is abstract. With images it stops being abstract, because you
can look at what came back.

Each digit is an 8×8 image — 64 numbers. Watch them rebuilt from fewer and fewer.

In [ ]:
picks = [np.where(DY == d)[0][0] for d in [0, 1, 3, 5, 8, 9]]
ks = [1, 2, 5, 10, 20, 64]
scaler = StandardScaler().fit(DX)        # same scaling as everywhere else

fig, axes = plt.subplots(len(ks) + 1, len(picks), figsize=(7, 8))
for j, idx in enumerate(picks):
    axes[0][j].imshow(DX[idx].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
axes[0][0].set_ylabel("original", fontsize=9)

for i, k in enumerate(ks, start=1):
    pk = PCA(k).fit(DZ)
    back = scaler.inverse_transform(pk.inverse_transform(pk.transform(DZ)))
    rec = back[picks]
    err = np.sqrt(((DX - back) ** 2).mean())
    for j in range(len(picks)):
        axes[i][j].imshow(rec[j].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[i][0].set_ylabel(f"{k} comp\n{pk.explained_variance_ratio_.sum():.0%}",
                          fontsize=9)
    print(f"{k:3d} components   keeps {pk.explained_variance_ratio_.sum():6.1%}"
          f"   error {err:.3f}")

for row in axes:
    for ax in row:
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

- **1 component** turns every digit into the same grey smear. 12% of the variance is
  nothing at all.
- **5 components** and you can read them. Five numbers standing in for sixty-four.
- **20 components** are almost exact — which is why the model scored the same as the full
  data.
- **64 components** give an error of exactly **0.000**. Keep every component and PCA
  throws nothing away; it has only rotated your data.

That last point is worth holding on to: **PCA itself is lossless.** The loss comes
entirely from *your* decision to drop the later components.

---
# 8 · The failure to watch for

PCA looks at **spread** and nothing else. In particular it never looks at your labels.

So what happens if the thing that separates your classes happens to have *small* spread?

In [ ]:
rg = np.random.default_rng(45)
n = 400
noise_col = rg.normal(0, 6.0, n)                       # huge spread, says nothing
label = rg.integers(0, 2, n)
signal_col = label * 1.2 + rg.normal(0, 0.28, n)       # tiny spread, says everything
X = np.c_[noise_col, signal_col]

pt = PCA().fit(X)
print("column standard deviations:", X.std(axis=0, ddof=1).round(3))
print("explained variance ratio  :", pt.explained_variance_ratio_.round(4))
print("PC1 loadings              :", pt.components_[0].round(4))

S = pt.transform(X)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
for c, col in zip([0, 1], [TEAL, CORAL]):
    axes[0].scatter(X[label == c, 0], X[label == c, 1], s=20, c=col,
                    edgecolors="white", linewidths=.4, label=f"class {c}")
    axes[1].scatter(S[label == c, 0], rg.normal(0, .05, (label == c).sum()),
                    s=20, c=col, edgecolors="white", linewidths=.4)
axes[0].legend(fontsize=9, frameon=False); axes[0].set_title("the data")
axes[1].set_ylim(-1, 1); axes[1].set_yticks([])
axes[1].set_title("keeping PC1 only"); axes[1].set_xlabel("PC1")
plt.tight_layout(); plt.show()

In [ ]:
a_pc1 = cross_val_score(LogisticRegression(max_iter=5000), S[:, [0]], label, cv=5).mean()
a_pc2 = cross_val_score(LogisticRegression(max_iter=5000), S[:, [1]], label, cv=5).mean()
a_all = cross_val_score(LogisticRegression(max_iter=5000), X, label, cv=5).mean()

print(f"keeping PC1 only  ({pt.explained_variance_ratio_[0]:.1%} of the spread): {a_pc1:.4f}")
print(f"keeping PC2 only  ({pt.explained_variance_ratio_[1]:.1%} of the spread): {a_pc2:.4f}")
print(f"keeping both columns                       : {a_all:.4f}")

**The component holding 98.8% of the spread scores 0.470 — worse than a coin toss. The
component holding 1.2% scores 0.980.**

Drop "the small components" here and you have deleted your model.

### But do not overlearn this

That example was built to be nasty. Most of the time PC1 *does* carry the signal — which
is the only reason PCA is useful at all. Check on real data:

In [ ]:
CZ = StandardScaler().fit_transform(load_breast_cancer().data)
CY = load_breast_cancer().target
CP = PCA().fit_transform(CZ)

print("accuracy from each single component, on breast cancer:")
for i in range(6):
    a = cross_val_score(LogisticRegression(max_iter=5000), CP[:, [i]], CY, cv=5).mean()
    print(f"  PC{i+1} alone: {a:.4f}")

Here **PC1 alone scores 0.912**, better than any other single component. That is the
normal case.

So the lesson is not "PCA is dangerous". It is:

> **Before you drop the small components, check that the thing you care about is not
> hiding in one of them.** One `cross_val_score` tells you.

---
# 9 · The payoff

Why bother at all? Because distance-based models have to touch every column, for every
prediction.

In [ ]:
rg2 = np.random.default_rng(7)
latent = rg2.normal(0, 1, (12000, 12))             # only 12 real dimensions
BIG = latent @ rg2.normal(0, 1, (12, 300)) + rg2.normal(0, .6, (12000, 300))
BIGY = (latent[:, 0] + 0.7 * latent[:, 1] > 0).astype(int)
Btr, Bte, bytr, byte_ = train_test_split(BIG, BIGY, test_size=.3, random_state=0)

import time
def best_of(f, reps=3):
    out, t = None, float("inf")
    for _ in range(reps):
        t0 = time.perf_counter(); out = f(); t = min(t, time.perf_counter() - t0)
    return out, t

m = KNeighborsClassifier(5, algorithm="brute").fit(Btr, bytr)
pred, t_full = best_of(lambda: m.predict(Bte))
print(f"300 columns : accuracy {accuracy_score(byte_, pred):.4f}   "
      f"predict {t_full:.3f}s")

bp, t_fit = best_of(lambda: PCA(12, random_state=0).fit(Btr))
mk = KNeighborsClassifier(5, algorithm="brute").fit(bp.transform(Btr), bytr)
Zte = bp.transform(Bte)
pred2, t_pca = best_of(lambda: mk.predict(Zte))
print(f" 12 columns : accuracy {accuracy_score(byte_, pred2):.4f}   "
      f"predict {t_pca:.3f}s")
print(f"\nspeedup {t_full / t_pca:.1f}x   (fitting PCA cost {t_fit:.3f}s, once)")
print(f"those 12 components hold {bp.explained_variance_ratio_.sum():.1%} of the spread")

Same accuracy, a few times faster on every prediction, and the PCA fit is paid once at
training.

### One honest footnote about that measurement

Notice `algorithm="brute"` in both lines. That is deliberate. Left on its defaults,
scikit-learn switches kNN to a tree-based search below about 15 columns — and on this many
query points the tree is *slower* than brute force. Run it both ways:

In [ ]:
ma = KNeighborsClassifier(5).fit(Btr, bytr)                 # default: algorithm="auto"
_, ta = best_of(lambda: ma.predict(Bte))
mb = KNeighborsClassifier(5).fit(bp.transform(Btr), bytr)
_, tb = best_of(lambda: mb.predict(Zte))

print(f"with algorithm='auto'  : 300 cols {ta:.3f}s   12 cols {tb:.3f}s"
      f"   -> {ta/tb:.1f}x")
print(f"  sklearn chose '{ma._fit_method}' for 300 columns"
      f" and '{mb._fit_method}' for 12")
print("\nOn defaults the PCA version comes out SLOWER -- not because PCA failed,")
print("but because we changed two things at once and only meant to change one.")

This is a benchmarking lesson more than a PCA lesson, and it is worth a minute of your
life: **when you compare two things, change one thing.**

---

# Recap

| | |
|---|---|
| **A component** | A direction through your data — one weight per original column, weights squaring to 1 |
| **PC1** | The direction of greatest spread. Maximises $w^TCw$; it is the top eigenvector of C |
| **λ (eigenvalue)** | The spread along that direction. The λ sum to the total spread |
| **Explained variance ratio** | $\lambda_i \div \sum\lambda$. Two columns → PC1 held 93.5%. Six columns → PC1+PC2 held 88.9% |
| **Loadings** | The weights, printed. How a component gets a name |
| **Scaling** | Mandatory. Raw wine gave PC1 = 99.8% proline, and accuracy fell 0.955 → 0.698 |
| **Choosing k** | A variance threshold picks a range; your model picks the number |
| **The blind spot** | PCA never sees labels. 1.2% of the spread held 0.980 accuracy |

### Three uses, in order of how often you should reach for them

1. **Look at data you could not plot.** Two components, one scatter plot. No modelling,
   just seeing.
2. **Remove columns that repeat each other.** Correlations of 0.85 mean you are feeding
   the model the same thing three times.
3. **Make the next model cheaper.** Same answer, less work per prediction.

**PCA is a lens, not a model.** It predicts nothing. It changes what you hand to the
thing that does.

---

Next: your homework is `Day45_Practice_Questions.ipynb` — twelve questions in four parts.